# CSM (lxing532/Dialogue-Topic-Segmenter) — 학습 (Colab + 로컬 Jupyter)

**환경 감지**: `[1]` 셀이 `IS_COLAB` 을 자동 감지. 로컬 Jupyter 에선
- `PROJECT_ROOT = os.getcwd()` (= Hi-OnTop repo root; notebook 을 repo root 에서 열기)
- `REPO_DIR = $PROJECT_ROOT/external/Dialogue-Topic-Segmenter`
- DailyDialog zip 은 `PROJECT_ROOT` 에 두면 `[2]` 가 자동 탐색
- ckpt 는 `.env` 의 `CSM_CKPT_DIR` (기본 `./checkpoints` = REPO_DIR/checkpoints)
- 커널 강제재시작 / Drive mount / files.upload / files.download 모두 Colab 에서만 동작

**실행 순서**
1. (선택) `[0]` GPU 확인
2. **`[1]`** 클론+의존성 설치
   - **Colab**: 셀 끝에서 커널 자동 재시작(정상). 재연결 후 `[1b]` 부터.
   - **Jupyter**: 재시작 안 함. 새 패키지 import 가 필요하면 Kernel ▸ Restart 후 `[1b]` 부터.
3. `[1b]` → `[2]`(DailyDialog) → 필요시 `[2b]`(Colab 업로드 / Jupyter 점검) → `[3]` 학습 → `[4]`/`[4b]` 평가 → `[5]` 번들.

**재현 등급 = 변형 (batch 32 → 64, 그 외 README 충실)**
- 학습: `bert-base-uncased` / **batch 64** / epochs 10 / margin 1 / AdamW lr=2e-5 eps=1e-8.
- 변경점: README 의 `-b 32` 를 **`-b 64`** 로 (사용자 결정 2026-05-20). 이유:
  A100 80GB 메모리 여유 + 학습시간 ~30h → ~15h 단축. codex 추정 Pk/WD/F1 영향 ±1pp.
- README 충실 재현으로 되돌리려면 `.env` 의 `CSM_BATCH=32` 만 바꿈.
- ⚠ `segment.py -m CM` 은 내부에서 backbone 을 `aws-ai/dse-bert-base` 로 하드코딩 → README train 의 `bert-base-uncased` ckpt 도 그대로 로드됨 (state_dict shape 호환).
- ⚠ **VAL 비활성** (`[2e]` 의 train.py): step 1000 마다 silent ckpt save 만, validation 미수행. validation 시간이 학습 시간만큼 소모되던 문제 해결. val 곡선은 평가 단 `[4]` 에서만 측정.
- `transformers==4.39.3` (Colab py3.12 tokenizers 휠 부재로 논문핀 4.27.4 불가). 4.39.3 도 `transformers.AdamW(<4.40)` 존재 → **train.py optimizer 호환**.
- 데이터: 원본 `ijcnlp_dailydialog.zip` (topic 포함) — Colab=`[2b]` 업로드 / Jupyter=`PROJECT_ROOT` 에 zip 둠.

**시각화 / 로그**
- tqdm bar: 5~10초마다 같은 줄 in-place 갱신 (`\r`) → "멈춤?" 즉시 식별.
- heartbeat log: 200 step (~85초) 마다 `tqdm.write()` 로 한 줄. timestamp + sie/총step + loss + it/s + epoch 경과/예상.
- epoch 끝 요약: 1줄 (avg_loss + gstep + 걸린 시간).
- `training_log.txt` 동시 누적.

> Runtime: GPU 권장 (A100 / T4+). 변형(DSE-BERT) 으로 가려면 `.env` 의 `CSM_ENCODER=aws-ai/dse-bert-base` 로만 바꾸면 됨.

In [ ]:
# [0] GPU / 환경 확인
import torch, sys
print('python', sys.version.split()[0])
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('!! GPU 없음 — Runtime>Change runtime type>GPU 로 바꾸고 재실행 권장 (CPU 학습은 매우 느림)')
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

In [ ]:
# [1] 클론 + 의존성 핀 설치
#   - Colab: 셀 끝에서 커널 강제재시작 (pip 반영). 재시작 후 [1b] 부터.
#   - Jupyter: 커널 강제재시작 안 함. 새 import 필요시 Kernel ▸ Restart 후 [1b].
#   ※ transformers 4.27.4 는 py3.12 tokenizers 휠 없음 → 4.39.3.
#     4.39.3 도 transformers.AdamW(<4.40) 존재 → train.py 무패치 재현.
import os
try:
    import google.colab as _gc  # noqa
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    PROJECT_ROOT = '/content'
    REPO_DIR = '/content/csm/Dialogue-Topic-Segmenter'
    os.makedirs(os.path.dirname(REPO_DIR), exist_ok=True)
else:
    # Jupyter: 노트북을 Hi-OnTop repo root 에서 열었다고 가정 (cwd = 그 dir)
    PROJECT_ROOT = os.path.abspath(os.getcwd())
    REPO_DIR = os.path.join(PROJECT_ROOT, 'external', 'Dialogue-Topic-Segmenter')
    os.makedirs(os.path.dirname(REPO_DIR), exist_ok=True)
print('IS_COLAB =', IS_COLAB)
print('PROJECT_ROOT =', PROJECT_ROOT)
print('REPO_DIR     =', REPO_DIR)

if not os.path.isdir(REPO_DIR):
    !git clone --depth 1 https://github.com/lxing532/Dialogue-Topic-Segmenter.git {REPO_DIR}
else:
    print('repo already cloned (skip git clone)')

# 핀: transformers 4.39.3 + python-dotenv (로컬/Colab 공통 .env 로딩).
# sentence-transformers 는 우리 NSP/CM eval 에 불필요 → 제외 (충돌 회피).
!pip -q install 'transformers==4.39.3' 'numpy<2' 'segeval==2.0.11' 'scikit-learn>=1.2' 'nltk==3.8.1' 'huggingface_hub>=0.20,<0.26' 'python-dotenv>=1.0' 2>&1 | tail -3

if IS_COLAB:
    print('\n==== 설치 완료 (Colab). 커널을 강제재시작합니다 (정상). ====')
    print('재시작 후: [1b] 셀부터 실행. [1] 은 다시 실행 금지.')
    os.kill(os.getpid(), 9)
else:
    print('\n==== 설치 완료 (Jupyter). 커널재시작 없이 진행. ====')
    print('새 패키지 import 충돌이 나면 Kernel ▸ Restart 후 [1b] 부터.')


In [ ]:
# [1b] 환경 변수 재설정 + 의존성 확인
#   [1] 이 Colab 에서 커널을 죽이므로 PROJECT_ROOT/REPO_DIR 다시 derive.
#   Jupyter 에선 [1] 의 값이 살아 있어도 동일 식이라 안전.
import os
try:
    import google.colab as _gc  # noqa
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    PROJECT_ROOT = '/content'
    REPO_DIR = '/content/csm/Dialogue-Topic-Segmenter'
else:
    PROJECT_ROOT = os.path.abspath(os.getcwd())
    REPO_DIR = os.path.join(PROJECT_ROOT, 'external', 'Dialogue-Topic-Segmenter')

assert os.path.isdir(REPO_DIR), f'[1] 을 먼저 실행(클론)하세요 — 없음: {REPO_DIR}'
os.chdir(REPO_DIR)
import torch, transformers, nltk
print('torch', torch.__version__, '| transformers', transformers.__version__,
      '| cuda', torch.cuda.is_available())
assert transformers.__version__ == '4.39.3', \
    f'transformers 4.39.3 필요(got {transformers.__version__}) — [1] 재실행/런타임 새로'
from transformers import AdamW  # noqa  (<4.40 → train.py 무패치)
print('transformers.AdamW OK → train.py 무패치 재현 (transformers 4.39.3)')
for pkg in ['stopwords', 'punkt']:
    try:
        nltk.download(pkg, quiet=True)
    except Exception as e:
        print('nltk warn', pkg, e)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('repo files:', sorted(os.listdir(REPO_DIR)))
print('eval datasets:', sorted(os.listdir(os.path.join(REPO_DIR, 'data', 'eval'))))


In [ ]:
# [2] DailyDialog 원본 준비 — PROJECT_ROOT 에서 zip 자동 탐색
#   Colab : [2b] 업로드 → /content/*.zip 발견
#   Jupyter: 같은 디렉터리(PROJECT_ROOT, 보통 Hi-OnTop repo root) 에 zip 두면 자동 탐색
#   원본 zip 안: ijcnlp_dailydialog/{dialogues_text,topic,act,emotion}.txt
import os, glob, zipfile, shutil
DD_DIR = os.path.join(REPO_DIR, 'data', 'train', 'dailydialog')
EXTRACT_DIR = os.path.join(PROJECT_ROOT, 'dd_extract')
os.makedirs(DD_DIR, exist_ok=True)
NEED = ['dialogues_text.txt', 'dialogues_topic.txt', 'dialogues_act.txt']

def have_dd():
    return all(os.path.isfile(os.path.join(DD_DIR, f)) and
               os.path.getsize(os.path.join(DD_DIR, f)) > 0 for f in NEED)

if not have_dd():
    raw = (glob.glob(os.path.join(PROJECT_ROOT, '**', 'ijcnlp_dailydialog*.zip'), recursive=True) +
           glob.glob(os.path.join(PROJECT_ROOT, '**', '*dailydialog*.zip'), recursive=True) +
           glob.glob(os.path.join(PROJECT_ROOT, '*.zip')))
    cand = []
    for p in dict.fromkeys(raw):
        if zipfile.is_zipfile(p):
            cand.append(p)
        else:
            print('무효 zip(삭제):', p, '(HTML/손상 — 진짜 원본 아님)')
            try: os.remove(p)
            except OSError: pass
    if cand:
        with zipfile.ZipFile(cand[0]) as z:
            z.extractall(EXTRACT_DIR)
        for zp in glob.glob(os.path.join(EXTRACT_DIR, '**', '*.zip'), recursive=True):
            if zipfile.is_zipfile(zp):
                with zipfile.ZipFile(zp) as z:
                    z.extractall(os.path.dirname(zp))
        for name in NEED + ['dialogues_emotion.txt']:
            hits = sorted(glob.glob(os.path.join(EXTRACT_DIR, '**', name), recursive=True),
                          key=lambda p: ('train' in p or 'valid' in p or 'test' in p, len(p)))
            if hits:
                shutil.copy(hits[0], os.path.join(DD_DIR, name))
                print('placed', name, '<-', hits[0])
    else:
        loc = '[2b] 업로드' if IS_COLAB else f'{PROJECT_ROOT}/ 에 진짜 원본 zip 두기'
        print(f'!! 유효한 ijcnlp_dailydialog.zip 없음 → {loc} 후 [2] 재실행')

if have_dd():
    shutil.copy(os.path.join(DD_DIR, 'dialogues_act.txt'),
                os.path.join(DD_DIR, 'dialogue_act.txt'))
    nl = {f: sum(1 for _ in open(os.path.join(DD_DIR, f))) for f in NEED}
    with open(os.path.join(DD_DIR, 'dialogues_topic.txt')) as fh:
        tline = fh.readline().strip()
    print('line counts:', nl, '| aligned:', len(set(nl.values())) == 1,
          '| topic int OK:', tline.isdigit(), '(sample=%r)' % tline)
    assert len(set(nl.values())) == 1 and tline.isdigit(), '원본 정렬/topic 형식 이상'
print('DailyDialog ready:', have_dd(), '| files:', sorted(os.listdir(DD_DIR)))


In [ ]:
# [2b] DailyDialog zip 업로드(Colab) / 확인(Jupyter)
#   Jupyter: 업로드 위젯 없음 — 같은 디렉터리(PROJECT_ROOT) 에 zip 두면 끝.
#            이 셀은 zip/txt 가 제자리에 있는지만 알려줌.
import os, shutil, glob
DD_DIR = os.path.join(REPO_DIR, 'data', 'train', 'dailydialog')
os.makedirs(DD_DIR, exist_ok=True)

if IS_COLAB:
    from google.colab import files
    up = files.upload()
    for fn in up:
        b = os.path.basename(fn)
        if b.endswith('.zip'):
            shutil.move(fn, os.path.join(PROJECT_ROOT, b))
            print('zip → %s/%s (이제 [2] 재실행)' % (PROJECT_ROOT, b))
        elif b.startswith('dialogues_') and b.endswith('.txt'):
            shutil.move(fn, os.path.join(DD_DIR, b))
            print('placed', b, '→', DD_DIR)
        else:
            shutil.move(fn, os.path.join(PROJECT_ROOT, b))
            print('saved %s/%s' % (PROJECT_ROOT, b))
    print('다음: [2] 셀 실행해서 "DailyDialog ready: True" 확인')
else:
    zips = sorted(glob.glob(os.path.join(PROJECT_ROOT, '*.zip')))
    txts = sorted(glob.glob(os.path.join(DD_DIR, 'dialogues_*.txt')))
    print('[2b] Jupyter 모드 — 업로드 위젯 없음.')
    print(f'  PROJECT_ROOT zips : {zips or "(없음)"}')
    print(f'  DD_DIR txts       : {txts or "(없음)"}')
    if zips:
        print('  → zip 있음. [2] 셀 실행 시 자동 추출됨.')
    elif txts:
        print('  → 이미 풀린 .txt 파일 있음. [2] 가 line count 확인할 것.')
    else:
        print(f'  → {PROJECT_ROOT}/ 에 ijcnlp_dailydialog*.zip 두고 [2] 재실행하세요.')


In [ ]:
# [2c] 레포 버그 수정 + forward 성능 패치 (학습 수식/데이터/HP/optimizer 불변)
#   (1) BUG FIX: segment.py 가 없는 coherence_model 을 import(모듈 최상단)
#       → NSP·CM eval 전부 ModuleNotFoundError. 클래스는 model_utils 에 존재
#       → coherence_model.py shim 생성. ckpt state_dict 키 동일 → strict load OK.
#       (neural_texttiling CM: text_encoder([[tok]*3]) → patched forward 와
#        shape/역할순서/loss소비 동일. eval/분포상 동등 성능최적화.
#   (2) PERF: CoherenceNet.forward 가 batch 를 파이썬 루프로 BERT 샘플×3 직렬
#       호출 → step당 72 forward → [3B,128] 1회. train dropout RNG
#       흐름 상이로 bitwise 아님 = 분포상 동등 성능최적화(codex).
#   ※ train.py 의 DataLoader/워커/resume 은 [2e] 가 train.py 전체를 소유하며
#     처리(여기서 train.py 안 건드림 — 중복/충돌 방지). 모두 멱등.
import re, os
assert 'REPO_DIR' in globals(), '[1b] 먼저 실행 (REPO_DIR 미정의)'

# (1) coherence_model.py shim — segment.py 의 missing import 해결
cm = os.path.join(REPO_DIR, 'coherence_model.py')
if os.path.exists(cm):
    print('coherence_model.py already exists (skip)')
else:
    open(cm, 'w').write('from model_utils import CoherenceNet\n')
    print('coherence_model.py shim 생성 (segment.py NSP/CM import 수정)')

# (2) batched CoherenceNet.forward (원본 per-sample 루프와 수치 동등)
mu = os.path.join(REPO_DIR, 'model_utils.py')
s = open(mu).read()
if '[perf] batched-equivalent' in s:
    print('model_utils.py already perf-patched (skip)')
else:
    NEW = '''    def forward(self, batch):
        # [perf] batched-equivalent of the original per-sample 3x BERT loop.
        B = len(batch)
        keys = list(batch[0][0].keys())
        big = {k: torch.cat([torch.cat([batch[i][r][k] for i in range(B)], 0)
                             for r in range(3)], 0).to(self.device)
               for k in keys}
        h = self.bert(**big).last_hidden_state[:, 0, :]      # [3B,768]
        dec = self.coherence_decoder(h)                      # [3B,2]
        sm = F.softmax(dec, dim=-1)                          # [3B,2]
        return sm.view(3, B, 2).permute(1, 0, 2).contiguous()  # [B,3,2]
'''
    s2 = re.sub(r"    def forward\(self, batch\):.*?return torch\.stack\(output, dim=0\)\n",
                NEW, s, count=1, flags=re.S)
    assert s2 != s and 'def forward' in s2, 'forward 패치 실패 — 원본 구조 확인'
    open(mu, 'w').write(s2)
    print('model_utils.py: batched forward (분포 동등 성능최적화)')
print('done — 알고리즘/데이터/HP/optimizer 불변. (DataLoader·resume 은 [2e])')

In [ ]:
# [2d] 체크포인트 경로
#   Colab : Drive mount 해서 /content/drive/MyDrive/csm_steprresume_v2 사용
#   Jupyter: .env 의 CSM_CKPT_DIR (기본 ./checkpoints = REPO_DIR/checkpoints)
import os
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    CKPT_DIR = '/content/drive/MyDrive/csm_steprresume_v2'   # 전용·신규
    os.makedirs(CKPT_DIR, exist_ok=True)
    _old = '/content/drive/MyDrive/csm_ckpts'
    if os.path.isdir(_old):
        print('※ 옛 폴더', _old, '존재 — 사용/삭제 안 함(직접 정리하셔도 됨).')
else:
    try:
        from dotenv import load_dotenv
        env_path = os.path.join(PROJECT_ROOT, '.env')
        if os.path.isfile(env_path):
            load_dotenv(env_path, override=False)
            print('.env loaded:', env_path)
    except ImportError:
        print('python-dotenv 미설치 — 셸 env 만 사용')
    raw = os.environ.get('CSM_CKPT_DIR', './checkpoints').strip()
    if os.path.isabs(raw):
        CKPT_DIR = raw
    else:
        # 상대경로 → REPO_DIR 기준 (.env 주석 정책)
        CKPT_DIR = os.path.normpath(os.path.join(REPO_DIR, raw))
    os.makedirs(CKPT_DIR, exist_ok=True)

_has = [f for f in os.listdir(CKPT_DIR) if f.endswith('.pth') or f == 'resume.pt']
print('CKPT_DIR =', CKPT_DIR,
      '| 기존 ckpt:', sorted(_has) if _has else '없음(→ fresh, 논문 등가 시작)')
print('[3] 이 이 경로만 사용.')


In [ ]:
# [2e] TRUE STEP-RESUME + log/val 최적화 — train.py 전체 재작성
#   - resume.pt: model+optimizer+scheduler+gstep+epoch+step_in_epoch+total
#   - epoch 마다 seed 로 결정적 셔플 → 멈췄던 'epoch 의 그 step'으로 정확히
#     점프(재계산 0). LR 스케줄 전 구간 단일·연속, 모멘텀 유지 = 논문 등가.
#   - cpt_<gstep>.pth = 순수 model state_dict (eval[4] 호환) 계속 저장.
#   ★ VAL disabled (속도/spam 최적화):
#       * validation 호출·val dataloader 생성 모두 제거. 전체 데이터를 train.
#       * ckpt save 는 SAVE_EVERY=1000 step 마다 silent 진행 (VAL 분리).
#   ★ heartbeat: 시간 기준 LOG_INTERVAL_SEC=60s (resume/fresh 무관 항상 동작).
#     loop 진입시 _t_last 를 -INTERVAL 로 초기화 → 첫 step 직후 즉시 첫 로그.
#     tqdm bar refresh: mininterval=5s, maxinterval=10s (\r in-place, 새 줄 X).
#   - DataLoader num_workers 포함([2c] train.py 패치 대체).
#   - seed42 = split+epoch-order+data_utils pseudo 결정화. HP/loss 불변. 멱등.
#   resume 호환: 기존 resume.pt 로드 후 새 LOG_EVERY/SAVE_EVERY 로 이어감.
import os, ast
assert 'REPO_DIR' in globals(), '[1b] 먼저 실행 (REPO_DIR 미정의)'

du = os.path.join(REPO_DIR, 'data_utils.py')
s = open(du).read()
if "[silenced]" not in s:
    s = s.replace(
        "            print('[Error] Problematic datapoint/dialogue, dropped it...')\n",
        "            globals()['_NDROP']=globals().get('_NDROP',0)+1  # [silenced]\n")
    open(du, 'w').write(s); print('data_utils: drop-print 침묵화')
else:
    print('data_utils already silenced (skip)')

TRAIN_PY = r'''import argparse, os, time, torch
from torch.utils.data import DataLoader, Subset
from transformers import AdamW, get_linear_schedule_with_warmup
from data_utils import UtteranceDataset
from model_utils import CoherenceNet
from transformers import AutoModel, AutoTokenizer
from tqdm import tqdm
import warnings
warnings.filterwarnings("ignore")
SEED = 42
LOG_INTERVAL_SEC = 60.0   # heartbeat log: 시간 기준. resume/fresh 무관 항상 동작.
SAVE_EVERY = 1000         # ckpt 저장 빈도 (step). VAL 과 분리, silent save.

def parse_args():
    p=argparse.ArgumentParser()
    p.add_argument("-t","--dataset",default="./data/train/dailydialog")
    p.add_argument("-r","--epochs",type=int,default=10)
    p.add_argument("-b","--batch_size",type=int,default=32)        # README default
    p.add_argument("-m","--margin",type=float,default=1)
    p.add_argument("-e","--text_encoder",default="bert-base-uncased")  # README default
    p.add_argument("-s","--checkpoints_path",default="./checkpoints/")
    return p.parse_args()

def collate_fn(b): return b

def marginal_ranking_loss(batch, margin):
    bt=batch[:, :, 0]
    l1=torch.nn.functional.relu(margin-(bt[:,0]-bt[:,1]))
    l2=torch.nn.functional.relu(margin-(bt[:,0]-bt[:,2]))
    l3=torch.nn.functional.relu(margin-(bt[:,1]-bt[:,2]))
    return torch.mean((l1+l2+l3)/3.0)

# (validation 함수 / val dataloader 전부 제거 — 학습 속도 우선)

def _save(model,opt,sch,gstep,epoch,sie,total,cps):
    torch.save(model.state_dict(), os.path.join(cps,"cpt_"+str(gstep)+".pth"))
    tmp=os.path.join(cps,"resume.pt.tmp")
    torch.save({"model":model.state_dict(),"opt":opt.state_dict(),
                "sched":sch.state_dict(),"gstep":gstep,"epoch":epoch,
                "sie":sie,"total":total}, tmp)
    os.replace(tmp, os.path.join(cps,"resume.pt"))

def epoch_order(n, epoch):
    g=torch.Generator(); g.manual_seed(SEED*10007+epoch)
    return torch.randperm(n, generator=g).tolist()

def train(model,full_train,opt,epochs,margin,device,cps,bs,nw):
    rp=os.path.join(cps,"resume.pt")
    bpe=(len(full_train)+bs-1)//bs            # batches per epoch
    if os.path.exists(rp):
        total=torch.load(rp,map_location="cpu")["total"]
    else:
        total=bpe*epochs
    sch=get_linear_schedule_with_warmup(opt,0,total)
    start_epoch=0; gstep=0; start_sie=0
    if os.path.exists(rp):
        ck=torch.load(rp,map_location=device)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"])
        sch.load_state_dict(ck["sched"])
        gstep=ck["gstep"]; start_epoch=ck["epoch"]; start_sie=ck["sie"]
        print("[resume] epoch %d/%d step_in_epoch %d gstep %d (target %d)"
              %(start_epoch+1,epochs,start_sie,gstep,total))
    else:
        print("[fresh] epoch1 step0 (target %d, 단일 LR 스케줄, seed %d)"
              %(total,SEED))
    print("[config] LOG_INTERVAL=%.0fs | SAVE_EVERY=%d step | VAL=disabled"
          %(LOG_INTERVAL_SEC, SAVE_EVERY))
    lf=open(os.path.join(cps, "training_log.txt"),"a")
    for ei in range(start_epoch,epochs):
        order=epoch_order(len(full_train),ei)
        sie0=start_sie if ei==start_epoch else 0
        sub=Subset(full_train, order[sie0*bs:])
        dl=DataLoader(sub,batch_size=bs,shuffle=False,collate_fn=collate_fn,
                      num_workers=nw,pin_memory=True)
        print("\n======== Epoch %d / %d (start step %d/%d) ========"
              %(ei+1,epochs,sie0,bpe))
        lf.write("\n==== Epoch %d/%d start_sie %d ====\n"%(ei+1,epochs,sie0))
        lf.flush(); tl=0.0; model.train()
        _t0 = time.time(); _t_last = _t0 - LOG_INTERVAL_SEC; _sie_last = sie0  # 첫 로그 즉시
        pbar = tqdm(enumerate(dl), total=len(dl), desc="Training",
                    mininterval=5.0, maxinterval=10.0,
                    dynamic_ncols=True, smoothing=0.1)
        for li,batch in pbar:
            sie=sie0+li
            model.zero_grad()
            out=model(batch); loss=marginal_ranking_loss(out,margin)
            tl+=loss.item(); loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
            opt.step(); sch.step(); gstep+=1
            # tqdm bar 의 우측에 [현재시각 loss it/s] 매 step set (mininterval=5s 마다 표시)
            _now=time.time()
            pbar.set_postfix_str("%s loss=%.4f"
                                 %(time.strftime("%H:%M:%S",time.localtime(_now)), loss.item()),
                                 refresh=False)
            # heartbeat log line — 시간 기준 (LOG_INTERVAL_SEC). resume/fresh 무관.
            if (_now - _t_last) >= LOG_INTERVAL_SEC:
                _d=_now-_t_last; _n=sie-_sie_last
                _ips=_n/_d if _d>0 else 0.0
                _total_d=_now - (_t0 - LOG_INTERVAL_SEC)   # 보정: _t0 초기화시 -INTERVAL 했던 부분
                _ts=time.strftime("%H:%M:%S",time.localtime(_now))
                _proj=bpe/(_ips if _ips>0 else 1)
                _msg=("[%s] e%d sie%d/%d gstep%d loss%.4f | %.1f it/s | epoch %.0fs / %.0fs (proj)"
                      %(_ts,ei+1,sie,bpe,gstep,loss.item(),_ips,_total_d,_proj))
                tqdm.write(_msg); lf.write(_msg+"\n"); lf.flush()
                _t_last=_now; _sie_last=sie
            # silent ckpt save — SAVE_EVERY 마다 (VAL 분리)
            if sie != 0 and sie % SAVE_EVERY == 0:
                _save(model,opt,sch,gstep,ei,sie+1,total,cps)
        _save(model,opt,sch,gstep,ei+1,0,total,cps)   # epoch 끝 → 다음 epoch
        _ep_t = time.time() - _t0
        _epoch_msg = "=== epoch %d done avg_loss %.4f gstep %d | %.0fs (%.1f min)" % (
            ei+1, tl/max(1,len(dl)), gstep, _ep_t, _ep_t/60.0)
        print(_epoch_msg); lf.write(_epoch_msg+"\n"); lf.flush()
    lf.close()
    print("[DONE] %d epoch 완료. 최종 gstep=%d"%(epochs,gstep))

def main():
    a=parse_args()
    import random
    random.seed(SEED); torch.manual_seed(SEED)
    device="cuda" if torch.cuda.is_available() else "cpu"
    enc=AutoModel.from_pretrained(a.text_encoder).to(device)
    tok=AutoTokenizer.from_pretrained(a.text_encoder)
    full=UtteranceDataset(os.path.join(a.dataset,"dialogues_text.txt"),
                          os.path.join(a.dataset,"dialogues_topic.txt"),
                          os.path.join(a.dataset,"dialogues_act.txt"),tok)
    # val split 자체를 안 만듦 (VAL disabled)
    g=torch.Generator(); g.manual_seed(SEED)
    perm=torch.randperm(len(full),generator=g).tolist()
    tr_idx=perm                                 # 전체를 train 으로 사용
    nw=min(8,(os.cpu_count() or 4))
    model=CoherenceNet(enc,device); model.to(device)
    opt=AdamW(model.parameters(),lr=2e-5,eps=1e-8)
    os.makedirs(a.checkpoints_path,exist_ok=True)
    train(model,Subset(full,tr_idx),opt,a.epochs,a.margin,device,
          a.checkpoints_path,a.batch_size,nw)

if __name__=="__main__":
    main()
'''
ast.parse(TRAIN_PY)
open(os.path.join(REPO_DIR,'train.py'),'w').write(TRAIN_PY)
print('train.py STEP-RESUME 재작성 완료: 그 epoch의 그 step으로 정확 점프, '
      '단일 LR, 재계산 0, eval 호환 cpt_*.pth, 워커, tqdm 10s. 알고리즘 불변.')

In [ ]:
# [3] CM coherence 학습 — README 충실 (encoder/batch/epochs/margin .env 구동)
#   README default (.env): -e bert-base-uncased -b 32 -r 10 -m 1
#   (README Step 2 의 train 명령 그대로. DSE-BERT 변형은 .env CSM_ENCODER 만 바꾸면 됨)
#   AdamW(lr2e-5,eps1e-8). transformers 4.39.3(<4.40, AdamW 존재).
#   override: 셸 env (또는 /content/.env) 의 CSM_ENCODER / CSM_BATCH / CSM_EPOCHS /
#     CSM_MARGIN — 없으면 README 기본. HF_TOKEN 있으면 huggingface-cli login.
#   첫 실행 [fresh] 단일 LR 스케줄 시작 → 멈췄다 재실행 시 [resume]
#   '그 epoch 의 그 step'으로 점프(재계산 0). CKPT_DIR(=[2d] 전용폴더)에
#   cpt_<gstep>.pth(순수 model, eval 호환) + resume.pt(full state) 저장.
#   '[DONE] N epoch 완료' = README 등가. (seed=42 로 split/order 고정·resume 가능)
import os
# .env 자동 로드. 우선순위: PROJECT_ROOT/.env → REPO_DIR/.env → ~/.env (+ Colab=/content/.env)
try:
    from dotenv import load_dotenv  # type: ignore
    cands = [os.path.join(PROJECT_ROOT, '.env'),
             os.path.join(REPO_DIR, '.env'),
             os.path.expanduser('~/.env')]
    if IS_COLAB:
        cands.insert(1, '/content/.env')
    for cand in cands:
        if os.path.isfile(cand):
            load_dotenv(cand, override=True)   # .env 값이 항상 우선 (kernel 안 새값 미반영 방지)
            print('.env loaded:', cand)
            break
except ImportError:
    pass  # dotenv 없으면 셸 env 만 사용

assert have_dd(), 'DailyDialog 원본 준비 안 됨 — [2]/[2b] 먼저'
ENCODER = os.environ.get('CSM_ENCODER', 'bert-base-uncased')  # README default
BATCH   = int(os.environ.get('CSM_BATCH', '32'))               # README default
EPOCHS  = int(os.environ.get('CSM_EPOCHS', '10'))              # README default
MARGIN  = float(os.environ.get('CSM_MARGIN', '1'))             # README default
HF_TOK  = os.environ.get('HF_TOKEN', '').strip()
if HF_TOK:
    os.environ['HUGGINGFACE_HUB_TOKEN'] = HF_TOK
    print('HF_TOKEN set (gated/private encoder 허용)')
print('HP:', dict(encoder=ENCODER, batch=BATCH, epochs=EPOCHS, margin=MARGIN))

try:
    CKPT_DIR                       # [2d] Drive 전용폴더
except NameError:
    CKPT_DIR = os.path.join(REPO_DIR, 'checkpoints')   # [2d] 미실행 시 로컬
os.makedirs(CKPT_DIR, exist_ok=True)
print('CKPT_DIR =', CKPT_DIR)
!python train.py -t {DD_DIR}/ -e {ENCODER} -s {CKPT_DIR} -m {MARGIN} -r {EPOCHS} -b {BATCH}

import glob, re
# resume.pt 제외 — cpt_<gstep>.pth 만, gstep 최대(최다학습) 선택
cks = glob.glob(os.path.join(CKPT_DIR, 'cpt_*.pth'))
def _step(p):
    m = re.search(r'cpt_(\d+)\.pth$', os.path.basename(p))
    return int(m.group(1)) if m else -1
cks = sorted([p for p in cks if _step(p) >= 0], key=_step)
print('\ncheckpoints:', [os.path.basename(c) for c in cks[-5:]],
      '(총', len(cks), ')')
assert cks, '체크포인트 미생성 — 위 학습 로그 확인'
BEST_CKPT = cks[-1]                # 최대 gstep = 가장 많이 학습된 순수 model
print('use BEST_CKPT =', BEST_CKPT, '(gstep=%d)' % _step(BEST_CKPT))

In [ ]:
# [4] 평가 — segment.py 로 NSP(zero-shot) + CM(학습 ckpt)
#   재현: CM = 학습된 coherence model (ckpt path 만 -e 인자로 전달).
#         ⚠ segment.py 내부에서 backbone = aws-ai/dse-bert-base 하드코딩 → README 의
#           bert-base-uncased ckpt 도 그대로 로드 (state_dict shape 호환).
#   NSP baseline = bert-base-uncased (README 의 NSP 옵션 중 하나).
import glob, re
EVAL_DIR = os.path.join(REPO_DIR, 'data', 'eval')
all_eval = {os.path.basename(p).lower(): p for p in glob.glob(os.path.join(EVAL_DIR, '*.json'))}
wanted = {}
for key, pat in [('dialseg711', 'dialseg'), ('tiage', 'tiage')]:
    hit = next((v for k, v in all_eval.items() if pat in k), None)
    if hit:
        wanted[key] = hit
print('eval targets:', {k: os.path.basename(v) for k, v in wanted.items()})

def run_segment(data_json, encoder, mode):
    """segment.py 실행 후 stdout 에서 Pk/WD/F1 파싱."""
    import subprocess
    out = subprocess.run(
        ['python', 'segment.py', '-t', data_json, '-e', encoder, '-m', mode],
        capture_output=True, text=True, cwd=REPO_DIR)
    txt = out.stdout + '\n' + out.stderr
    def grab(label):
        m = re.search(label + r'[^0-9-]*([0-9]*\.?[0-9]+)', txt, re.I)
        return float(m.group(1)) if m else None
    res = {'pk': grab('P_?k'), 'wd': grab('WindowDiff|Windiff|WD'), 'f1': grab('F1')}
    if all(v is None for v in res.values()):
        print(f'--- segment.py 출력 파싱 실패 ({mode}) ---\n', txt[-1500:])
    return res

rows = []
for ds, path in wanted.items():
    rows.append(('NSP (zero-shot)', ds, run_segment(path, 'bert-base-uncased', 'NSP')))
    rows.append(('CM (trained, 논문 본방법)', ds, run_segment(path, BEST_CKPT, 'CM')))
for label, ds, r in rows:
    print(f'{ds:11s} {label:24s} Pk={r["pk"]} WD={r["wd"]} F1={r["f1"]}')

In [ ]:
# [4b] TextTiling 베이스라인 (nltk) — 같은 eval json 에 동일 Pk/WD/F1 공식
from nltk.tokenize import TextTilingTokenizer
from nltk.metrics import pk as nltk_pk, windowdiff as nltk_wd
from sklearn.metrics import f1_score
import numpy as np, json

def load_eval(path):
    data = json.load(open(path))
    items = data if isinstance(data, list) else data.get('data', data)
    out = []
    for d in (items.values() if isinstance(items, dict) else items):
        utts = d['utterances']
        segs = set(d.get('segments', d.get('segment', [])))
        yt = [1 if i in segs else 0 for i in range(len(utts))]
        if yt: yt[-1] = 0
        if len(utts) >= 2: out.append((utts, yt))
    return out

def official_pk_wd(yt, yp):
    n_seg = sum(yt) + 1
    k = max(2, int(round(len(yt) / n_seg / 2)))
    ts, ps = ''.join(map(str, yt)), ''.join(map(str, yp))
    return float(nltk_pk(ts, ps, k=k)), float(nltk_wd(ts, ps, k=k))

def texttiling_pred(utts):
    tt = TextTilingTokenizer(w=10, k=6)
    try:
        tiles = tt.tokenize('\n\n'.join(u.strip() or '.' for u in utts))
    except Exception:
        return [0] * len(utts)
    pred = []
    for t in tiles:
        lines = [x for x in t.strip().split('\n\n') if x != '']
        if not lines: continue
        pred += [0] * len(lines)
        pred[-1] = 1
    pred = (pred + [0] * len(utts))[:len(utts)]
    if pred: pred[-1] = 0
    return pred

for ds, path in wanted.items():
    dia = load_eval(path)
    pks, wds, g, p = [], [], [], []
    for utts, yt in dia:
        yp = texttiling_pred(utts)
        a, b = official_pk_wd(yt, yp); pks.append(a); wds.append(b)
        g += yt; p += yp
    r = {'pk': float(np.mean(pks)), 'wd': float(np.mean(wds)),
         'f1': float(f1_score(g, p, zero_division=0))}
    rows.append(('TextTiling (nltk)', ds, r))
    print(f'{ds:11s} TextTiling (nltk)  Pk={r["pk"]:.3f} WD={r["wd"]:.3f} F1={r["f1"]:.3f}')

In [ ]:
# [5] 결과 표 + ckpt 묶기 (Colab 만 자동 다운로드)
import datetime, shutil, json, torch, transformers, os
TR_VER = transformers.__version__
REPRO = 'README-faithful (bert-base-uncased train) — transformers ' + TR_VER
step_n = ''.join(ch for ch in os.path.basename(BEST_CKPT) if ch.isdigit())
lines = ['# CSM 재현 — lxing532 CM / NSP / TextTiling\n',
         f'date: {datetime.date.today()} | encoder(CM): {ENCODER} | epochs(target): {EPOCHS} '
         f'| batch: {BATCH} | margin: {MARGIN} | device: {DEVICE}\n',
         f'재현등급: {REPRO}. ckpt=cpt_{step_n} (누적 step={step_n}). '
         f'transformers={TR_VER}(AdamW 무패치,<4.40) / torch={torch.__version__}. '
         'DailyDialog = 원본 ijcnlp_dailydialog (topic 포함) verbatim.\n',
         '※ step-resume(opt+sched+gstep+epoch 복원, seed=42 pseudo/split/order 결정화) '
         '-> 단일 연속 LR Nep 등가, 그 epoch/step 정확점프. perf(forward 배치화: '
         'train dropout RNG 상이 = 분포상 동등). '
         '레포 seed 미고정 → 비결정성. metric Pk/WD↓ F1↑ (NSP/CM=segment.py, TextTiling=nltk).\n',
         '| method | dataset | Pk ↓ | WD ↓ | F1 ↑ |',
         '|---|---|---:|---:|---:|']
def fmt(x): return f'{x:.3f}' if isinstance(x, (int, float)) else str(x)
for label, ds, r in rows:
    lines.append(f'| {label} | {ds} | {fmt(r["pk"])} | {fmt(r["wd"])} | {fmt(r["f1"])} |')
results_md = '\n'.join(lines) + '\n'

BUNDLE = os.path.join(PROJECT_ROOT, 'csm_artifacts')
RESULTS_MD = os.path.join(PROJECT_ROOT, 'results.md')
os.makedirs(BUNDLE, exist_ok=True)
open(RESULTS_MD, 'w').write(results_md)
print(results_md)

shutil.copy(BEST_CKPT, os.path.join(BUNDLE, 'csm_cm_' + os.path.basename(BEST_CKPT)))
shutil.copy(RESULTS_MD, os.path.join(BUNDLE, 'results.md'))
with open(os.path.join(BUNDLE, 'meta.json'), 'w') as f:
    json.dump({'repo': 'lxing532/Dialogue-Topic-Segmenter',
               'repro_grade': REPRO,
               'transformers': TR_VER, 'torch': torch.__version__,
               'transformers_note': 'paper pin 4.27.4 not installable on py3.12 → 4.39.3 (AdamW present, unpatched)',
               'perf_patch': 'batched CoherenceNet.forward + DataLoader workers (math-identical, no learning change)',
               'training_mode': 'TRUE step-resume: resume.pt(model+opt+sched+gstep+epoch+sie); single continuous LR; seed=42 (split+epoch-order+data_utils pseudo) -> exact epoch/step jump, no recompute ~= single continuous N-epoch run.',
               'cumulative_step': step_n,
               'encoder_CM': ENCODER, 'epochs_target': EPOCHS, 'batch': BATCH,
               'margin': MARGIN, 'optimizer': 'transformers.AdamW lr2e-5 eps1e-8 (unpatched)',
               'dailydialog': 'original ijcnlp_dailydialog (verbatim, topic incl.)',
               'seed': 'NOT fixed by repo (run-to-run variance)',
               'ckpt': os.path.basename(BEST_CKPT)}, f, indent=2)
zip_path = shutil.make_archive(os.path.join(PROJECT_ROOT, 'csm_artifacts'), 'zip', BUNDLE)
print('bundle:', zip_path, '(', round(os.path.getsize(zip_path)/1e6, 1), 'MB )')

if IS_COLAB:
    try:
        from google.colab import files
        files.download(zip_path)
    except Exception as e:
        print('자동 다운로드 불가:', e)
else:
    print('(Jupyter) 로컬 파일 그대로 사용:', zip_path)


## 가져온 뒤 (로컬 Hi-OnTop)

`csm_artifacts.zip` 안 `csm_cm_*.pth` 가 학습된 CM 체크포인트.

1. `csm_cm_*.pth` 를 로컬 `outputs/runs/_misc/` (또는 지정 경로)에 둠
2. 로컬에서 이 ckpt 로 **superseg + tiage + dialseg711** 을 Hi-OnTop 공식 SuperDialseg Pk/WD/F1 harness 로 재평가 → 기존 `outputs/experiments/.../REPORT.md` 와 동일 metric 으로 통합 (Claude 가 처리)
3. `results.md` 는 Colab 단계 sanity 표 (metric provenance 섞임 + topic 합성 → 최종 비교는 로컬 공식 수치 기준, REPORT 에 topic 합성 한계 명시)